<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 37px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
  Machine Learning<br>
  </div> 

<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 35px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
  -<br>
  </div> 
  
<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 35px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
      Illustration with Random Forests<br><br>
  </div> 

<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 25px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
      <font color=orange> 4 - Classification</font>
  </div>

<a id="plan"></a>



### Classification with Random Forests

- [Categorical target](#categorical_target)
- [Evaluation criteria of binary classifiers](#binary_classifier_scoring)
- [Decision Tree Classifier](#tree_classifier)
- [Random Forest Classifier](#random_forest_classifier)
- [Model tuning](#model_tuning)
- [Random forests versus generic ensemble methods](#ensemble_methods)
- [Precision - Recall curve and threshold selection](#rec_prc_curve)
- [Model calibration](#model_calibration)
- [Multiclass classification](#multiclass_classification)


[Bottom](#bottom)

### Imports

In [ ]:
# for package auto reload
%load_ext autoreload
%autoreload 2

# for better rendering of plots in jupyter notebook
%matplotlib inline

In [ ]:
# base modules
from pathlib import Path

# for manipulating data
import numpy as np
import pandas as pd

# for visualization
from matplotlib import pyplot as plt
from sklearn import metrics
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.calibration import CalibratedClassifierCV

# for Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree

# custom utility code
from eclyon.plot import draw_tree
from eclyon.transforms import process_df, split_vals

#  Classification with Random Forests

[Back to top](#plan)

We shall change our target variable from log sale prices to a set of _categorical sale prices_. Then we will use the `RandomForestClassifier` model to compute prediction. Classifiers are typically used in order to predict one class among a finite set of possibilities, however these models actually generate a *probability estimate* over the full set of possibilities. The resulting prediction is then simply taken as the most likely class given this output probability estimate.


Let us introduce the use of classifiers on out dataset, with the continuous variable *SalePrice* converted into a categorical one.

In [ ]:
path_to_repo = Path().resolve().parent
path_to_data = path_to_repo / "data"

In [ ]:
df_transformed = pd.read_feather(path_to_data / "transformed")

df, y, nas = process_df(df_transformed, "SalePrice")

### $\bullet$ Categorical target

[Back to top](#plan)

We shall change our target variable from log sale prices to a set of _categorical sale prices_, and we will recast the sale Price prediction problem as a _binary classification_ problem:

In [ ]:
fig = pd.Series(y).plot(kind="density", bw_method=0.05)

# alternatively
# sns.set_style('whitegrid')
# sns.kdeplot(np.array(y_train), bw_method = 0.05)

In [ ]:
pd.Series(y).plot(kind="density", bw_method=0.01)

In [ ]:
threshold = 10.5
y_cat = np.empty(len(y), dtype=object)
y_cat[y<threshold] = "Cheap"
y_cat[y>=threshold] = "Expensive"
print(y_cat)
print(np.unique(y_cat, return_counts=True))

In [ ]:
n_total = len(df)
n_valid = 12000  # same as Kaggle's test set size
n_train = n_total - n_valid
n_small = 20000

X_train, X_val = split_vals(df, n_train)
y_train, y_val = split_vals(y_cat, n_train)

X_small, _ = split_vals(df, n_small)
y_small, _ = split_vals(y_cat, n_small)

print("Number of small training data points: X = {}, y = {}".format(X_small.shape, y_small.shape))
print("Number of full training data points: X = {}, y = {}".format(X_train.shape, y_train.shape))
print("Number of validation data points: X = {}, y = {}".format(X_val.shape, y_val.shape))

In [ ]:
X_train.head(3)

In [ ]:
len(y_train), y_train

<a id='binary_classifier_scoring'></a>

### $\bullet$ Evaluation criteria of binary classifiers

[Back to top](#plan)

See the `Evaluation criteria of binary classifiers` section of the companion documentation `docs/machine-learning/supervized-classification.md`.

Note : The generic evaluation setting is for 0-1 valued models. Here we set the class y=0 (the "irrelevant category") as "Cheap" and the class y=1 (the "relevant category") as "Expensive".

In [ ]:
def base_classif_perf(y_true, y_pred, neg_label, pos_label) -> None:
    """
    Evaluate the performance of a binary classification model.
    """   
    print("Accuracy : {:.3f}".format(metrics.accuracy_score(y_true, y_pred)))
    print("Recall : {:.3f}".format(metrics.recall_score(y_true, y_pred, pos_label=pos_label)))
    print("Precision : {:.3f}".format(metrics.precision_score(y_true, y_pred, pos_label=pos_label)))
    print("F1 score : {:.3f}".format(metrics.f1_score(y_true, y_pred, pos_label=pos_label)))
    return

#### Exercice

We want to predict if a patient is sick (y=1) or healthy (y=0).
We have a set of 10 patients : y_true = [1, 0, 0, 0, 0, 1, 0, 1, 0, 0].
- Your first model predicts all patients as healthy : y_pred_1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0].
- Your second model predicts : y_pred_2 = [1, 0, 0, 0, 0, 0, 0, 0, 0, 0].
- Your 3rd model predicts : y_pred_3 = [1, 0, 1, 0, 0, 1, 0, 0, 0, 0].
- Your 4th model predicts : y_pred_4 = [1, 0, 1, 0, 0, 1, 1, 1, 1, 1].

Compute all scoring metrics for the 3 models.
Which model do you choose ?

<a id='tree_classifier'></a>

### $\bullet$ Decision Tree Classifier

[Back to top](#plan)

Implementing a Decision Tree Classifier is similar to implementing a Decision Tree Regressor, except on:<br>

- The Hererogeneity criterion, specified with the `criterion` option
- The possible imbalance of importance of each category, specified with the `class_weight` option

See the `Decision tree classifier` section of the companion documentation `docs/machine-learning/supervized-classification-tree-based.md`.

In [ ]:
tree_classifier = DecisionTreeClassifier(
    class_weight=None,  # specific to classifiers
    criterion="gini",  # specific to classifiers
    splitter="best",
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=None,
)

In [ ]:
%time tree_classifier.fit(X_small, y_small)

print(tree_classifier.score(X_small, y_small))
print(tree_classifier.score(X_val, y_val))

# this brings a more challenging quality evaluation
base_classif_perf(y_true=y_val, y_pred=tree_classifier.predict(X_val), neg_label="Cheap", pos_label="Expensive")

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(15, 10), dpi=300)
plot_tree(
    tree_classifier,
    feature_names=X_train.columns,
    class_names=tree_classifier.classes_,
    filled=True,
)
plt.show()
    

In [ ]:
tree_features = [k for k, v in zip(tree_classifier.feature_names_in_, tree_classifier.feature_importances_) if v]
index = 0

X_val[tree_features].take([index])

In [ ]:
pbs = tree_classifier.predict_proba(X_val.take([index]))[0]
cls = tree_classifier.predict(X_val.take([index]))[0]
pbs, cls

In [ ]:
807 / (807 + 388), 388 / (807 + 388)

In [ ]:
pbs = tree_classifier.predict_proba(X_val.take(range(10))).tolist()
cls = tree_classifier.predict(X_val.take(range(10))).tolist()
list(zip(pbs, cls))

#### Exercices

1. Compute the predicted probas and classification manually:

In [ ]:
X_val[tree_features].take([11])

# compute these values by hand:
# pbs = tree_classifier.predict_proba(X_val.take([11]))[0]
# cls = tree_classifier.predict(X_val.take([11]))[0]
# pbs, cls

2. Implement by yourself the computation of:
- `accuracy`
- `recall`
- `precision`
- `specificity`
- `fpr`
- `f1-score`

Check that your new `classif_perf` matches with `base_classif_perf` on the metrics in common.

In [ ]:
def accuracy_score(y_true, y_pred) -> float:
    """
    Computes the proba that the prediction is equal to the label.
    """
    return sum(y_true == y_pred) / len(y_true)


def recall_score(y_true, y_pred, neg_label, pos_label) -> float:
    '''
    Computes the proba that the prediction is 1 knowing that the label is 1.
    '''
    y__1 = [(true, pred) for true, pred in zip(y_true, y_pred) if true == pos_label]
    y_11 = [(true, pred) for true, pred in y__1 if pred == pos_label]
    recall  = (0 if len(y__1) == 0 else len(y_11)/len(y__1))
    return recall


def precision_score(y_true, y_pred, neg_label, pos_label) -> float:
    '''
    Computes the proba that the label is 1 knowing that the prediction is 1.
    '''
    return 0.


def specificity_score(y_true, y_pred, neg_label, pos_label) -> float:
    '''
    Computes the proba that the label is 0 knowing that the prediction is 0.
    '''
    return 0.


def fpr_score(y_true, y_pred, neg_label, pos_label) -> float:
    '''
    Computes the proba that the prediction is 1 knowing that the label is 0.
    '''
    return 0.


def f1_score(y_true, y_pred, neg_label, pos_label) -> float:
    '''
    Computes the f1-score.
    '''
    return 0.


def classif_perf(y_true, y_pred, neg_label, pos_label) -> None:
    """
    Evaluate the performance of a binary classification model.
    """   
    print("Accuracy : {:.3f}".format(accuracy_score(y_true, y_pred)))
    print("Recall : {:.3f}".format(recall_score(y_true, y_pred, neg_label, pos_label)))
    print("Precision : {:.3f}".format(precision_score(y_true, y_pred, neg_label, pos_label)))
    print("Specificity: {:.3f}".format(specificity_score(y_true, y_pred, neg_label, pos_label)))
    print("False Positive Rate: {:.3f}".format(fpr_score(y_true, y_pred, neg_label, pos_label)))
    print("F1 score : {:.3f}".format(f1_score(y_true, y_pred, neg_label, pos_label)))
    return

In [ ]:
classif_perf(y_true=y_val, y_pred=tree_classifier.predict(X_val), neg_label="Cheap", pos_label="Expensive")

<a id='random_forest_classifier'></a>

### $\bullet$ Random Forest Classifier

[Back to top](#plan)

Now, we will grow a **forest of independant trees**, each one focusing on a different representation of the dataset (bootstrapping of the instances, subset of the features).

At the end the final prediction will be a **vote** (mean of predictions) among all the trees in forest.

See the `Random forest classifier` section of the companion documentation `docs/machine-learning/supervized-classification-tree-based.md`.

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=20,
    class_weight=None,  # classifier specific
    criterion="gini",  # classifier specific
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=42,
    bootstrap=True,
    oob_score=True,
    max_samples=None,
    warm_start=False,
    n_jobs=-1,
    verbose=0,
)

In [ ]:
%time classifier.fit(X_small, y_small)

print(classifier.score(X_small, y_small))  # classification accuracy on training set
print(classifier.score(X_val, y_val))  # classification accuracy on validation set

classif_perf(y_true=y_val, y_pred=classifier.predict(X_val), neg_label="Cheap", pos_label="Expensive")

In [ ]:
pbs = classifier.predict_proba(X_val.take([0]))[0]
cls = classifier.predict(X_val.take([0]))[0]
pbs, cls

In [ ]:
# The predicted label is 'Expensive' precisely when its estimated probability is > 0.5
cat_mapping = {0: "Cheap", 1: "Expensive"}
probas_1 = classifier.predict_proba(X_val)[:, 1]
prob_labels = [cat_mapping[int(p > 0.5)] for p in probas_1]
pred_labels = classifier.predict(X_val)

(sum(prob_labels == pred_labels) / len(pred_labels)).item()

In [ ]:
trees = classifier.estimators_

In [ ]:
fig, axes = plt.subplots(nrows = 2, ncols = 1, figsize = (6, 10), dpi = 300)
plot_tree(
    trees[0], 
    feature_names=X_train.columns,
    class_names=tree_classifier.classes_,
    max_depth=2,
    filled=True,
    ax=axes[0],
)
axes[0].set_title("Tree 1")
plot_tree(
    trees[1],
    feature_names=X_train.columns, 
    class_names=tree_classifier.classes_,
    max_depth=2,
    filled=True,
    ax=axes[1],
)
axes[1].set_title("Tree 2")
plt.show()

In [ ]:
# all tree predictions on single data point
point = X_val.take([0]).values

forest_pbs = classifier.predict_proba(X_val.take([0]))[0]
trees_pbs = np.asarray([tree.predict_proba(point)[0] for tree in trees])

avg = trees_pbs.sum(0)
avg = avg / sum(avg)

print("Forest probability:\n", forest_pbs)
print("Tree probabilities:\n", trees_pbs)
print("\nAverage probability:\n", avg)

### $\bullet$ Model tuning

[Back to top](#plan)

Let's improve the performance of our model, first playing with hyperparameters by hand, then using grid-search cross-validation

In [ ]:
# Increase tree depth
classifier = RandomForestClassifier(
    n_estimators = 20, 
    class_weight = None, # classifier specific
    criterion = 'gini',  # classifier specific
    max_depth = None, 
    min_samples_split = 3, 
    min_samples_leaf = 2, 
    min_weight_fraction_leaf = 0.0, 
    max_features = None,
    max_leaf_nodes = None, 
    min_impurity_decrease = 0.0,
    bootstrap = True, 
    oob_score = True, 
    max_samples = 10000,
    random_state = 42,
)

In [ ]:
%time classifier.fit(X_small, y_small)

In [ ]:
print(classifier.score(X_small, y_small))  # classification accuracy on training set
print(classifier.score(X_val, y_val))  # classification accuracy on validation set
print(classifier.oob_score_)  # classification accuracy on oob samples

base_classif_perf(y_true=y_val, y_pred=classifier.predict(X_val), neg_label="Cheap", pos_label="Expensive")

This model is conservative as it has a strong tendency to _underestimate_ the price of a bulldozer, since only 72.6% of high-priced bulldozers are predicted so (the recall is 72.6%). Nonetheless, when the model predicts that a bulldozer is high-priced, this predictions is reasonably reliable since it has 91.2% change to be right (the precision is 91.2%).

In [ ]:
# Grid search evaluates each combination of params set in the param grid:
# - using the specified base estimator
# - by computing cross-validation score over `cv` splits of the data
# - using the defined scoring method
rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [20, 50, 75],
    'max_depth': [5, 20, None],
    'min_samples_split': [2, 5, 15],
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,              
    scoring=metrics.make_scorer(metrics.f1_score, pos_label='Expensive'),
    verbose=2,
)

grid_search.fit(X_small, y_small)

In [ ]:
print("Best parameters:", grid_search.best_params_)
print("Best F1 score:", grid_search.best_score_)

In [ ]:
report = pd.DataFrame(grid_search.cv_results_).sort_values(by="mean_test_score", ascending=False)
report.head()

In [ ]:
# Let's train the best model on full dataset !
%time grid_search.best_estimator_.fit(X_train, y_train)

In [ ]:
classif_perf(y_true=y_val, y_pred=grid_search.best_estimator_.predict(X_val), neg_label="Cheap", pos_label="Expensive")

#### Exercices

What is the meaning of the different params, see the [scikit-learn page](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html):

- `n_estimators`
- `max_depth` 
- `min_samples_split`
- `min_samples_leaf`
- `min_weight_fraction_leaf`
- `max_features`
- `max_leaf_nodes`
- `min_impurity_decrease`
- `bootstrap`
- `oob_score` => How is it computed according to the [reference doc](https://scikit-learn.org/stable/auto_examples/ensemble/plot_ensemble_oob.html#sphx-glr-auto-examples-ensemble-plot-ensemble-oob-py) ?
- `random_state`: Controls both the randomness of the bootstrapping of the samples used
    when building trees (if ``bootstrap=True``) and the sampling of the
    features to consider when looking for the best split at each node
    (if ``max_features < n_features``).
- `class_weight`
- `max_samples`

### $\bullet$ Random forests versus generic ensemble methods

[Back to top](#plan)

Propose answers to these questions:

- How does RF differ from tree bagging ?
- How does RF differ from tree stacking ?
- How does RF differ from tree voting ?

### $\bullet$ Precision - Recall curve and threshold selection

[Back to top](#plan)


Most of Scikit-learn classifiers not only transform a data point into a predicted class, but in fact into a probability of membership over the full set of classes. In the case of binary classifiers, the predicted label $\widehat{c} \in \{0, 1\}$ is deduced from this probability $\widehat{p}$ over $\{0, 1\}$ by comparing the probability attributed to the label 1 with the reference threshold 0.5 :

$$ \widehat{c} = 1 \; \Longleftrightarrow \;  \widehat{p}(1) \geqslant 0.5 $$


This threshold is natural, but can be changed if for instance one wants to counter-balance the effect of an unbalanced training dataset, or wants to favor recall (resp. precision) at the cost of precision (resp. recall). Indeed, selecting a threshold equal to 0.0 yields a (rather trivial) classifier that will always predicts label 1, so with low precision, and conversely selecting the threshold equal to 1.0 yields an (equally trivial) classifier that always predics the label 0, so with zero recall. By continuously moving this threshold from 0.0 to 1.0, one gets a classifier that continuously evolve from a low-precision model towards a low-recall model, with a certain threshold yielding optimal precision-recall scores lying somewhere in between.

The evolution of the precision and recall scores of the binary classifier obtained by continuously shifting the threshold from 0.0 to 1.0 is calld the **Precision-Recall curve**.

**Remark** _The binary classifier is trained once and for all prior to computing its precision-recall curve!_ The fact of having precision and recall scores that change is only caused by varying the threshold serving to convert probabilistic predictions into deterministic predictions.

In [ ]:
def precision_recall_curve_with_threshold(clf, X, y, threshold=0.5):
    class_names = classifier.classes_.tolist()
    probas = clf.predict_proba(X)[:, 1]
    targets = [class_names.index(t) for t in y]
    precision, recall, thresholds = metrics.precision_recall_curve(targets, probas)

    # calculate precision-recall AUC
    auc_val = metrics.auc(recall, precision)
    print("Area Under Curve (AUC) : {:.3f}".format(auc_val))
    # plot no skill
    if threshold is not None:
        index = len([y for y in thresholds if y <= threshold])  # thresholds are sorted

        print("Recall with threshold = {}: {:.2f}%".format(threshold, recall[index] * 100))
        print("Precision with threshold = {} : {:.2f}%".format(threshold, precision[index] * 100))

        plt.plot([0, 1], [precision[index], precision[index]], linestyle="--", color="grey")
        plt.plot([recall[index], recall[index]], [0, 1], linestyle="--", color="grey")

        # plot the precision-recall curve for the model
        plt.plot(recall[:index], precision[:index], color="navy", marker=".")
        plt.plot(recall[index:], precision[index:], color="orange", marker=".")

        # show the plot
        plt.title("Precision-Recall curve with threshold = {}".format(threshold))
        plt.xlabel(r"Recall $P(\, \widehat{1}\, | 1)$")
        plt.ylabel(r"Precision $P(\, 1\, | \widehat{1})$                                ", rotation="horizontal")
    else:
        # plot the precision-recall curve for the model
        plt.plot(recall, precision, color="orange", marker=".")

        # show the plot
        plt.title("Precision-Recall curve")
        plt.xlabel(r"Recall $P(\, \widehat{1}\, | 1)$")
        plt.ylabel(r"Precision $P(\, 1\, | \widehat{1})$                                ", rotation="horizontal")

    plt.show()
    return

In [ ]:
# The predicted label is 1 precisely when its estimated probability is >= 0.5
threshold = 0.5

probas_1 = classifier.predict_proba(X_val)[:, 1]
prob_labels = [int(p > threshold) for p in probas_1]

pred_labels = classifier.predict(X_val)

sum(prob_labels == pred_labels) / len(pred_labels)

In [ ]:
# beware that y axis is along [0.5, 1]
rec_prc_curve = metrics.PrecisionRecallDisplay.from_estimator(classifier, X_val, y_val)
rec_prc_curve

In [ ]:
classif_perf(classifier, X_val, y_val, "Expensive")

In this situation a threshold of 0.35 seems to yield a more balanced classifier :

In [ ]:
precision_recall_curve_with_threshold(classifier, X_val, y_val, threshold=0.35)

In [ ]:
precision_recall_curve_with_threshold(classifier, X_val, y_val, threshold=0.6)

In [ ]:
precision_recall_curve_with_threshold(classifier, X_val, y_val, threshold=0.4)

The Area Under Curve (AUC) is a score between 0.0 (worst) and 1.0 (best), which is the area of the zone located below the precision-recall curve (e.g. the AUC is the integral of this curve).<br>
It is a measure of quality of the underlying probabilistic classifier, but does not qualify the quality of the resulting deterministic classifier, whatever the chosen threshold.

### $\bullet$ Model calibration

[Back to top](#plan)


Manually selecting the threshold in order to balance precision and recall can be dangerous for _uncalibrated_ classifiers, that is, classifiers where determinisitc predictions change a lot for a small change of threshold :

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=20,
    class_weight="balanced",  # classifier specific
    criterion="gini",  # classifier specific
    max_depth=25,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=42,
    bootstrap=True,
    oob_score=True,
    max_samples=None,
    warm_start=False,
    n_jobs=-1,
    verbose=0,
)

In [ ]:
%time classifier.fit(X_small, y_small)

print(classifier.score(X_small, y_small))  # classification accuracy on smalling set
print(classifier.score(X_val, y_val))  # classification accuracy on validation set
print(classifier.oob_score_)  # classification accuracy on oob samples

In [ ]:
probas = classifier.predict_proba(X_val)[:, 1]

precision_recall_curve_with_threshold(y_val, probas, threshold=0.35)

Shifting the classification threshold from 0.4 to 0.6 induces a 28% precision jump and a 34% recall jump !

Let us now try with a _calibrated_ Random Forest :

In [ ]:
calibrated_classifier = CalibratedClassifierCV(
    estimator=classifier,
    method="sigmoid",  # 'sigmoid' or 'isotonic'
    ensemble=True,
    cv=8,
    n_jobs=-1,
)

In [ ]:
%time calibrated_classifier.fit(X_small, y_small)

print(calibrated_classifier.score(X_small, y_small))  # classification accuracy on smalling set
print(calibrated_classifier.score(X_val, y_val))  # classification accuracy on validation set

In [ ]:
y_predict_valid = calibrated_classifier.predict(X_val)

acc = metrics.accuracy_score(y_val, y_predict_valid)
rec = metrics.recall_score(y_val, y_predict_valid)
prc = metrics.precision_score(y_val, y_predict_valid)
spe = specificity_score(y_val, y_predict_valid)
f1 = metrics.f1_score(y_val, y_predict_valid)

print("Accuracy : {:.2f}%".format(acc * 100))
print("Recall : {:.2f}%".format(rec * 100))
print("Precision : {:.2f}%".format(prc * 100))
print("Specificity : {:.2f}%".format(spe * 100))
print("F1-score : {:.2f}%".format(f1 * 100))

In [ ]:
probas = calibrated_classifier.predict_proba(X_val)[:, 1]

precision_recall_curve_with_threshold(y_val, probas, threshold=0.35)

The calibrated version of the Random Forest Classifier has more balanced precision and recall scores. These scores are moreover more stable with respect to the chosen threshold, as shifting the classification threshold from 0.4 to 0.6 induces this time a reduced 9% precision jump and 18% recall jump.

#### Calibration for probability interpretation as confidence levels

We use here calibration as a mean to reduce variance in the resulting deterministic classifiers when shifting the decision threshold. Originally, this method was designed in order to get probabilistic models whose _probability estimates truly correspond to confidence levels_, in the sense that the region of input points that were assigned a probability of $p$ to be of a given label should form a region where the empirical ratio of points with this label is $p$.<br>
For more details see the [scikit-learn documentation](https://scikit-learn.org/stable/modules/calibration.html#calibration)

### $\bullet$ Multiclass classification

[Back to top](#plan)


Let us replicate the process followed for binary classification to the case of 5-fold classification

#### Data categorization

In [ ]:
intervals_5 = pd.cut(y, bins=5)
intervals_5[:10].tolist()

In [ ]:
label_encoder = LabelEncoder()

y_cat_5 = label_encoder.fit_transform(intervals_5)
y_cat_5[:10]

In [ ]:
label_encoder.classes_.tolist()

In [ ]:
y_cat_5_train, y_cat_5_valid = split_vals(y_cat_5, n_train)
y_cat_5_small, _ = split_vals(y_cat_5, n_small)

y_cat_5_train.shape, y_cat_5_valid.shape, y_cat_5_small.shape

#### Model training

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=20,
    class_weight="balanced",  # classifier specific
    criterion="gini",  # classifier specific
    max_depth=25,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=42,
    bootstrap=True,
    oob_score=True,
    max_samples=None,
    warm_start=False,
    n_jobs=-1,
    verbose=0,
)


calibrated_classifier = CalibratedClassifierCV(
    estimator=classifier,
    method="sigmoid",  # 'sigmoid' or 'isotonic'
    ensemble=True,
    cv=5,
    n_jobs=-1,
)

In [ ]:
%time classifier.fit(X_small, y_cat_5_small)

print(classifier.score(X_small, y_cat_5_small))  # classification accuracy on training set
print(classifier.score(X_val, y_cat_5_valid))  # classification accuracy on validation set

In [ ]:
%time calibrated_classifier.fit(X_small, y_cat_5_small)

print(calibrated_classifier.score(X_small, y_cat_5_small))  # classification accuracy on training set
print(calibrated_classifier.score(X_val, y_cat_5_valid))  # classification accuracy on validation set

#### Evaluation

We remind that intuitively, precision is the ability of a binary classifier not to label as positive a sample that is negative, and recall is the ability of the classifier to find all the positive samples.

Evaluation of a multiclass classifier can, in addition to computing the overall accuracy, either be computed using  `macro` average or `weighted` macro-average of precision, recall and F1-score :

- macro : the macro-average of the recall score is obtained by taking the average, over all classes $c$, of the ratio of well-classified points among points with true class $c$, whereas the precision score is obtained by taking the average, over all classes $c$, of the ratio of well-classified points among points with predicted class $c$. The macro-averaged F1-score is taken by applying the F1-score formula to the macro-averages of precision and recall scores.


- weighted macro-average : Same as macro, but averages over classes are ponderated by the ratio of points belonging to each class. This permits to take into account the dataset imbalance over the different classes : scores computed over well-represented classes have more weight on the resulting scores, whereas poorly-represented classes have little impact on the resulting scores. This gives a more realistic evaluation since future point sets to classify will typically follow the same imbalance as observed during evaluation.


For _multi-label_ classification there is also the notion of `micro` average as alternative to macro and weighted-macro averages.

Macro-averages :

In [ ]:
y_predict_valid = classifier.predict(X_val)

average = "macro"

acc = metrics.accuracy_score(y_cat_5_valid, y_predict_valid)
rec = metrics.recall_score(y_cat_5_valid, y_predict_valid, average=average)
prc = metrics.precision_score(y_cat_5_valid, y_predict_valid, average=average)
f1 = metrics.f1_score(y_cat_5_valid, y_predict_valid, average=average)

print("Accuracy : {:.2f}%".format(acc * 100))
print("Recall : {:.2f}%".format(rec * 100))
print("Precision : {:.2f}%".format(prc * 100))
print("F1-score : {:.2f}%".format(f1 * 100))

In [ ]:
y_predict_valid = calibrated_classifier.predict(X_val)

average = "macro"

acc = metrics.accuracy_score(y_cat_5_valid, y_predict_valid)
rec = metrics.recall_score(y_cat_5_valid, y_predict_valid, average=average)
prc = metrics.precision_score(y_cat_5_valid, y_predict_valid, average=average)
f1 = metrics.f1_score(y_cat_5_valid, y_predict_valid, average=average)

print("Accuracy : {:.2f}%".format(acc * 100))
print("Recall : {:.2f}%".format(rec * 100))
print("Precision : {:.2f}%".format(prc * 100))
print("F1-score : {:.2f}%".format(f1 * 100))

Weighted macro-averages :

In [ ]:
y_predict_valid = classifier.predict(X_val)

average = "weighted"

acc = metrics.accuracy_score(y_cat_5_valid, y_predict_valid)
rec = metrics.recall_score(y_cat_5_valid, y_predict_valid, average=average)
prc = metrics.precision_score(y_cat_5_valid, y_predict_valid, average=average)
f1 = metrics.f1_score(y_cat_5_valid, y_predict_valid, average=average)

print("Accuracy : {:.2f}%".format(acc * 100))
print("Recall : {:.2f}%".format(rec * 100))
print("Precision : {:.2f}%".format(prc * 100))
print("F1-score : {:.2f}%".format(f1 * 100))

In [ ]:
y_predict_valid = calibrated_classifier.predict(X_val)

average = "weighted"

acc = metrics.accuracy_score(y_cat_5_valid, y_predict_valid)
rec = metrics.recall_score(y_cat_5_valid, y_predict_valid, average=average)
prc = metrics.precision_score(y_cat_5_valid, y_predict_valid, average=average)
f1 = metrics.f1_score(y_cat_5_valid, y_predict_valid, average=average)

print("Accuracy : {:.2f}%".format(acc * 100))
print("Recall : {:.2f}%".format(rec * 100))
print("Precision : {:.2f}%".format(prc * 100))
print("F1-score : {:.2f}%".format(f1 * 100))

The different scores are better summarized in the following report :

In [ ]:
y_predict_valid = classifier.predict(X_val)

report = metrics.classification_report(
    y_cat_5_valid,
    y_predict_valid,
    target_names=[str(interval) for interval in label_encoder.classes_.tolist()],
)
print(report)

In [ ]:
y_predict_valid = calibrated_classifier.predict(X_val)

report = metrics.classification_report(
    y_cat_5_valid,
    y_predict_valid,
    target_names=[str(interval) for interval in label_encoder.classes_.tolist()],
)
print(report)

#### Confusion matrix

Ultimately, classification preformances are better understood though the _confusion matrix_ of the model on the evaluation set:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

metrics.ConfusionMatrixDisplay.from_estimator(
    classifier,
    X_val,
    y_cat_5_valid,
    display_labels=[str(interval) for interval in label_encoder.classes_.tolist()],
    include_values=True,
    xticks_rotation=45,
    values_format=None,
    cmap="viridis",
    ax=ax,
    colorbar=True,
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

metrics.ConfusionMatrixDisplay.from_estimator(
    calibrated_classifier,
    X_val,
    y_cat_5_valid,
    display_labels=[str(interval) for interval in label_encoder.classes_.tolist()],
    include_values=True,
    xticks_rotation=45,
    values_format=None,
    cmap="viridis",
    ax=ax,
    colorbar=True,
)

[Back to top](#plan)